In [1]:
from transformers import Sam3Processor, Sam3Model
import torch
from PIL import Image
import requests

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

# Load image
image_url = "http://images.cocodataset.org/val2017/000000077595.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

# Segment using text prompt
inputs = processor(images=image, text="ear", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process results
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,
    mask_threshold=0.5,
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"Found {len(results['masks'])} objects")
# Results contain:
# - masks: Binary masks resized to original image size
# - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
# - scores: Confidence scores


/media/volume/Chau/miniconda3/envs/sam/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1468/1468 [00:00<00:00, 4437.59it/s]


Found 2 objects


In [2]:
model

Sam3Model(
  (vision_encoder): Sam3VisionModel(
    (backbone): Sam3ViTModel(
      (embeddings): Sam3ViTEmbeddings(
        (patch_embeddings): Sam3ViTPatchEmbeddings(
          (projection): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        )
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (layer_norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (layers): ModuleList(
        (0-31): 32 x Sam3ViTLayer(
          (layer_norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (rotary_emb): Sam3ViTRotaryEmbedding()
          (attention): Sam3ViTRoPEAttention(
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (o_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm2): LayerNorm((1024,

In [3]:
import torch
import time
from collections import defaultdict

# ── config ──────────────────────────────────────────────────────────────────
WARMUP = 3
RUNS   = 10

# ── CUDA-event timers attached via hooks ────────────────────────────────────
latencies = defaultdict(list)   # name -> list of ms
_start_events = {}

def make_hooks(name):
    def pre_hook(module, input):
        if device == "cuda":
            e = torch.cuda.Event(enable_timing=True)
            e.record()
            _start_events[name] = e
        else:
            _start_events[name] = time.perf_counter()

    def post_hook(module, input, output):
        if device == "cuda":
            end = torch.cuda.Event(enable_timing=True)
            end.record()
            torch.cuda.synchronize()
            elapsed = _start_events[name].elapsed_time(end)   # ms
        else:
            elapsed = (time.perf_counter() - _start_events[name]) * 1e3
        latencies[name].append(elapsed)

    return pre_hook, post_hook

handles = []
for name, module in model.named_children():
    pre, post = make_hooks(name)
    handles.append(module.register_forward_pre_hook(pre))
    handles.append(module.register_forward_hook(post))

# ── benchmark loop ───────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    for i in range(WARMUP + RUNS):
        if device == "cuda":
            torch.cuda.synchronize()
        _ = model(**inputs)
        if device == "cuda":
            torch.cuda.synchronize()
        if i < WARMUP:
            for v in latencies.values():   # discard warmup
                v.clear()

# ── remove hooks ─────────────────────────────────────────────────────────────
for h in handles:
    h.remove()

# ── results table ────────────────────────────────────────────────────────────
import statistics, math

total_mean = sum(statistics.mean(v) for v in latencies.values())

print(f"{'Component':<30} {'Mean (ms)':>10} {'Std (ms)':>9} {'% total':>8}")
print("-" * 62)
for name, vals in latencies.items():
    mean = statistics.mean(vals)
    std  = statistics.stdev(vals) if len(vals) > 1 else 0.0
    pct  = 100.0 * mean / total_mean if total_mean > 0 else 0.0
    print(f"{name:<30} {mean:>10.2f} {std:>9.2f} {pct:>7.1f}%")
print("-" * 62)
print(f"{'TOTAL (sum of components)':<30} {total_mean:>10.2f}")


Component                       Mean (ms)  Std (ms)  % total
--------------------------------------------------------------
vision_encoder                     338.24      0.02    84.6%
text_encoder                        11.93      0.29     3.0%
text_projection                      0.11      0.11     0.0%
detr_encoder                        22.87      0.07     5.7%
detr_decoder                        18.84      0.29     4.7%
dot_product_scoring                  0.46      0.02     0.1%
mask_decoder                         7.16      0.02     1.8%
--------------------------------------------------------------
TOTAL (sum of components)          399.61


In [4]:
import torch
import time
from collections import defaultdict
from transformers import Sam3VideoProcessor

# ── config ───────────────────────────────────────────────────────────────────
NUM_FRAMES = 8    # number of video frames to simulate
WARMUP     = 3
RUNS       = 10

# ── build video input (replicate image as N frames) ──────────────────────────
video_processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

frames = [image] * NUM_FRAMES     # replace with real video frames if available
video_inputs = video_processor(images=frames, return_tensors="pt").to(device)
print(f"pixel_values shape: {video_inputs['pixel_values'].shape}")   # (T, C, H, W)

# broadcast text inputs to match frame batch size
input_ids_video      = inputs["input_ids"].expand(NUM_FRAMES, -1)
attention_mask_video = inputs["attention_mask"].expand(NUM_FRAMES, -1) if "attention_mask" in inputs else None

# ── hook-based CUDA-event profiler ───────────────────────────────────────────
video_latencies = defaultdict(list)
_start_events   = {}

def make_hooks(name, store):
    def pre_hook(module, input):
        if device == "cuda":
            e = torch.cuda.Event(enable_timing=True)
            e.record()
            _start_events[name] = e
        else:
            _start_events[name] = time.perf_counter()

    def post_hook(module, input, output):
        if device == "cuda":
            end = torch.cuda.Event(enable_timing=True)
            end.record()
            torch.cuda.synchronize()
            elapsed = _start_events[name].elapsed_time(end)
        else:
            elapsed = (time.perf_counter() - _start_events[name]) * 1e3
        store[name].append(elapsed)

    return pre_hook, post_hook

handles = []
for name, module in model.named_children():
    pre, post = make_hooks(name, video_latencies)
    handles.append(module.register_forward_pre_hook(pre))
    handles.append(module.register_forward_hook(post))

# ── benchmark loop ────────────────────────────────────────────────────────────
model.eval()
for i in range(WARMUP + RUNS):
    if device == "cuda":
        torch.cuda.synchronize()
    with torch.no_grad():
        _ = model(
            pixel_values=video_inputs["pixel_values"],
            input_ids=input_ids_video,
            attention_mask=attention_mask_video,
        )
    if device == "cuda":
        torch.cuda.synchronize()
    if i < WARMUP:
        for v in video_latencies.values():
            v.clear()

for h in handles:
    h.remove()

# ── compare image vs video ────────────────────────────────────────────────────
import statistics

img_lat     = latencies   # defined in previous profiling cell
video_total = sum(statistics.mean(v) for v in video_latencies.values())
image_total = sum(statistics.mean(v) for v in img_lat.values()) if img_lat else None

print(f"\nVideo: {NUM_FRAMES} frames")
print(f"{'Component':<30} {'Image (ms)':>11} {'Video (ms)':>11} {'Ratio':>8}")
print("-" * 65)
for name, vvals in video_latencies.items():
    vmean = statistics.mean(vvals)
    imean = statistics.mean(img_lat[name]) if name in img_lat else float("nan")
    ratio = vmean / imean if imean > 0 else float("nan")
    print(f"{name:<30} {imean:>11.2f} {vmean:>11.2f} {ratio:>7.2f}x")
print("-" * 65)
img_str = f"{image_total:>11.2f}" if image_total else f"{'N/A':>11}"
print(f"{'TOTAL':<30} {img_str} {video_total:>11.2f}")


pixel_values shape: torch.Size([8, 3, 1008, 1008])

Video: 8 frames
Component                       Image (ms)  Video (ms)    Ratio
-----------------------------------------------------------------
vision_encoder                      338.24     2469.06    7.30x
text_encoder                         11.93       17.88    1.50x
text_projection                       0.11        0.11    0.99x
detr_encoder                         22.87      159.00    6.95x
detr_decoder                         18.84       51.05    2.71x
dot_product_scoring                   0.46        0.47    1.02x
mask_decoder                          7.16       30.86    4.31x
-----------------------------------------------------------------
TOTAL                               399.61     2728.43


In [5]:
from capture_qkvx_sam3 import capture_features, scan_global_layers, plot_sobel_heatmaps

features = capture_features(model, processor, image)
sobel_maps = scan_global_layers(features)

# with original image alongside (3 columns: input | heatmap | overlay)
plot_sobel_heatmaps(sobel_maps, image=image, save_path="sobel_global_layers.png")

# heatmap only
plot_sobel_heatmaps(sobel_maps, cmap="inferno")


ModuleNotFoundError: No module named 'capture_qkvx_sam3'